In [2]:
import os

import pandas as pd
import numpy as np

import regex as re
from datetime import datetime

In [14]:
data_path = "../data/in/"
output_path = "../data/out/"

regions = {"vor": "20241214-0617_gtfs_vor_2024", #vienna, lower austria, burgenland
           "ooevv": "20241212-0156_gtfs_ooevv_2024", #upper austria
           "esg": "20241203-0058_gtfs_esg_2024", #linz
           "verbundlinie": "20241217-0310_gtfs_verbundlinie_2024", #styria
           "kaernterlinien": "20241214-0253_gtfs_kaerntnerlinien_2024", #carinthia
           "salzburgverkehr": "20241217-0359_gtfs_salzburgverkehr_2024", #salzburg
           "vvt": "20241217-0436_gtfs_vvt_2024", #tyrol
           "vmobil": "20241212-0624_gtfs_vmobil_2024", #vorarlberg
           "obb": "GTFS_2024_obb"} #oebb maybe 20241217-0222_gtfs_evu_2024

# select region
state_name = "esg"
# select day for calculation in format YYYYMMDD in 2024
selected_day = 20240131

# stop categories
table_roman = np.array([
    ["I", "I", "II", "III"],        # < 5 min
    ["I", "II", "III", "III"],      # 5 >= x <= 10
    ["II", "III", "IV", "IV"],      # 10 < x < 20
    ["III", "IV", "V", "V"],        # 20 >= x < 40
    ["IV", "V", "VI", "VI"],        # 40 >= x <= 60
    ["V", "VI", "VII", "VII"],      # 60 < x <= 120  
    ["X", "VII", "VIII", "VIII"],    # 120 < x <= 210 
    ["X", "X", "X", "X"],               # > 210 
                                    # X = empty, i.e. worst case
])
table = np.array([
    [0, 0, 0, 0],        # < 5 min
    [0, 1, 2, 2],        # 5 >= x <= 10
    [1, 2, 3, 3],        # 10 < x < 20
    [2, 3, 4, 4],        # 20 >= x < 40
    [3, 4, 5, 5],        # 40 >= x <= 60
    [4, 5, 6, 6],        # 60 < x <= 120  
    [-1, 6, 7, 7],       # 120 < x <= 210 
    [-1, -1, -1, -1],    # > 210
])

# transport_category = ["Fernverkehr REX", 
#                       "S-Bahn / U-Bahn, Regionalbahn, Schnellbus, Lokalbahn", 
#                       "Straßenbahn, Metrobus, 0-Bus", 
#                       "Bus"]
route_type_translation = {0: 2, 1: 1, 2: 0, 3: 3, 11: 3,} #7: 3, 4: 3



In [15]:
def lookup_category(interval, t_cat):
    """
    Lookup the stop category based on the interval and the transport type.
    """
    if interval < 5:
        return table[0][t_cat]
    elif interval <= 10:
        return table[1][t_cat]
    elif interval < 20:
        return table[2][t_cat]
    elif interval < 40:
        return table[3][t_cat]
    elif interval <= 60:
        return table[4][t_cat]
    elif interval <= 120:
        return table[5][t_cat]
    elif interval <= 210:
        return table[6][t_cat]
    else:
        return table[7][t_cat]
    
def category_to_roman(t_cat, reverse=False):
    """
    Convert a category to a roman numeral or vice versa if reverse is True
    """
    roman_numerals = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "X"]
    if reverse:
        return roman_numerals.index(t_cat)
    return roman_numerals[t_cat]
    
def detect_route_type(trip_name, route_type):
    """
    Detect the type of transportation based on the trip name and the route type
    By default, the transport type is only based on the route type.
    For route type 2 (train), the trip_name is used, to further distinguish between different trains, if available.
    """
    # TODO: also consider route_short/long_name ??? 
    if route_type == 2 and not pd.isna(trip_name):
        trips_name = trip_name.lower()
        if any(x in trips_name for x in ["rj", "rjx", "nj", "en", "ic", "ec", "ice", "ecb", "rex", "wb", "rgj", "cjx" ]): #fernverkehr
            return 0
        else:
            return 1
    else:
        return route_type_translation[route_type]

In [16]:
# read in the data
path = f"{data_path}/{regions[state_name]}/"

stops = pd.read_csv(path + "/stops.txt", quotechar='"', sep=",")
stop_times = pd.read_csv(path + "/stop_times.txt", quotechar='"', sep=",")
trips = pd.read_csv(path + "/trips.txt", quotechar='"', sep=",")
routes = pd.read_csv(path + "/routes.txt", quotechar='"', sep=",")
calendar = pd.read_csv(path + "/calendar.txt", quotechar='"', sep=",")
calendar_dates = pd.read_csv(path + "/calendar_dates.txt", quotechar='"', sep=",")

print(stops.shape)
stops.head()

(1248, 9)


,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
0,at:44:40002:0:0,Linz/Donau Saporoshjestraße,48.251829,14.322757,NaN,NaN,Pat:44:40002,Level 0,0.0
1,at:44:40002:0:1,Linz/Donau Saporoshjestraße,48.251686,14.322874,NaN,NaN,Pat:44:40002,Level 0,1.0
2,at:44:40002:0:2,Linz/Donau Saporoshjestraße,48.251817,14.322586,NaN,NaN,Pat:44:40002,Level 0,2.0
3,at:44:40002:0:3,Linz/Donau Saporoshjestraße,48.251805,14.322703,NaN,NaN,Pat:44:40002,Level 0,3.0
4,at:44:40005:0:0,Linz/Donau Aichinger,48.341945,14.307584,NaN,NaN,Pat:44:40005,Level 0,0.0


In [24]:
# find weekday of selected day
# days = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
# date = datetime.fromisoformat(str(selected_day))
# day_string = days[date.weekday()]

date = datetime.fromisoformat(str(selected_day))
day_string = date.strftime("%A").lower()

print(day_string)

calendar_filtered = calendar[(calendar['start_date'] <= selected_day) & (calendar['end_date'] >= selected_day) & (calendar[day_string] == 1)].copy()
#calendar_dates_filtered = calendar_dates[calendar_dates["date"] == selected_day]

added_service = calendar_dates[(calendar_dates['date'] == selected_day) & (calendar_dates['exception_type'] == 1)]
removed_service = calendar_dates[(calendar_dates['date'] == selected_day) & (calendar_dates['exception_type'] == 2)]

# only keep services that run on that weekday
# calendar_filtered = calendar_filtered[calendar_filtered[day_string] == 1]

print(calendar_filtered.shape)
removed_service.head()

wednesday
(24, 10)


,service_id,date,exception_type
1875,TA+ek000#1,20240131,2
2580,TA+jp000#1,20240131,2
3277,TA+pr400,20240131,2
3697,TA+rq500,20240131,2


In [ ]:
# keep only valid trips
trips_filtered = trips[trips['service_id'].isin(calendar_filtered['service_id'])]
print(trips_filtered.shape)

# remove services with exception_type = 2 from calendar_dates
#trips_filtered = trips_filtered[trips_filtered["service_id"].isin(calendar_dates_filtered[calendar_dates_filtered["exception_type"] == 2]["service_id"])]
trips_filtered = trips_filtered[~trips_filtered["service_id"].isin(removed_service["service_id"])]
print(trips_filtered.shape)

# add services with exception_type = 1 from calendar_dates
trips_full = pd.concat([trips_filtered, trips[trips["service_id"].isin(added_service["service_id"])]])
print(trips_full.shape)
trips_full.head()

if trips_full.shape[0] == 0:
    print("No trips found! Most likely because no service is running on the selected day or the input data is not complete.")

(857, 8)
(737, 8)
(737, 8)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
966,at:esg:101:,TA+go800,1.TA.2-101-E-c24-1.1.R,2-101-E-c24-1.1.R,Linz/Donau Urnenhain Urfahr,NaN,1,NaN
967,at:esg:101:,TA+io800,102.TA.2-101-E-c24-1.1.R,2-101-E-c24-1.1.R,Linz/Donau Urnenhain Urfahr,NaN,1,NaN
968,at:esg:101:,TA+io800,105.TA.2-101-E-c24-1.1.R,2-101-E-c24-1.1.R,Linz/Donau Urnenhain Urfahr,NaN,1,NaN
969,at:esg:101:,TA+go800,108.TA.2-101-E-c24-1.2.H,2-101-E-c24-1.2.H,Linz/Donau Aichinger,NaN,0,NaN
970,at:esg:101:,TA+go800,11.TA.2-101-E-c24-1.1.R,2-101-E-c24-1.1.R,Linz/Donau Urnenhain Urfahr,NaN,1,NaN


In [19]:
# merge trips with routes information
routes_trips = pd.merge(trips_full, routes, on='route_id', how='left')
display(routes_trips)

# keep only valid trips route_type, remove e.g funicular 7 and ferry 4
routes_trips = routes_trips[routes_trips['route_type'].isin([0, 1, 2, 3, 11])]
# TODO: check if this is correct

# translate route type
routes_trips['trip_short_name'] = routes_trips['trip_short_name'].astype('str')
routes_trips['rank'] = routes_trips.apply(lambda x: detect_route_type(x['trip_short_name'], x['route_type']), axis=1)

print(routes_trips.shape)
routes_trips[routes_trips['rank'].isna()]

,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id,agency_id,route_short_name,route_long_name,route_type
0,at:esg:101:,TA+go800,1.TA.2-101-E-c24-1.1.R,2-101-E-c24-1.1.R,Linz/Donau Urnenhain Urfahr,NaN,1,NaN,1,101,Linz Urnenhain Urfahr - Linz Aichinger,3
1,at:esg:101:,TA+io800,102.TA.2-101-E-c24-1.1.R,2-101-E-c24-1.1.R,Linz/Donau Urnenhain Urfahr,NaN,1,NaN,1,101,Linz Urnenhain Urfahr - Linz Aichinger,3
2,at:esg:101:,TA+io800,105.TA.2-101-E-c24-1.1.R,2-101-E-c24-1.1.R,Linz/Donau Urnenhain Urfahr,NaN,1,NaN,1,101,Linz Urnenhain Urfahr - Linz Aichinger,3
3,at:esg:101:,TA+go800,108.TA.2-101-E-c24-1.2.H,2-101-E-c24-1.2.H,Linz/Donau Aichinger,NaN,0,NaN,1,101,Linz Urnenhain Urfahr - Linz Aichinger,3
4,at:esg:101:,TA+go800,11.TA.2-101-E-c24-1.1.R,2-101-E-c24-1.1.R,Linz/Donau Urnenhain Urfahr,NaN,1,NaN,1,101,Linz Urnenhain Urfahr - Linz Aichinger,3
...,...,...,...,...,...,...,...,...,...,...,...,...
732,at:esg:77:,TA+bl700,17.TA.2-77-E-c24-1.4.H,2-77-E-c24-1.4.H,Linz/Donau JKU I Universität Nord,NaN,0,NaN,1,77,Linz Hauptbahnhof - Linz Universität,3
733,at:esg:77:,TA+bl700,2.TA.2-77-E-c24-1.3.R,2-77-E-c24-1.3.R,Linz/Donau Hauptbahnhof,NaN,1,NaN,1,77,Linz Hauptbahnhof - Linz Universität,3
734,at:esg:77:,TA+bl700,20.TA.2-77-E-c24-1.4.H,2-77-E-c24-1.4.H,Linz/Donau JKU I Universität Nord,NaN,0,NaN,1,77,Linz Hauptbahnhof - Linz Universität,3
735,at:esg:77:,TA+bl700,5.TA.2-77-E-c24-1.3.R,2-77-E-c24-1.3.R,Linz/Donau Hauptbahnhof,NaN,1,NaN,1,77,Linz Hauptbahnhof - Linz Universität,3


(737, 13)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id,agency_id,route_short_name,route_long_name,route_type,rank


In [20]:
# prepare stops and stop_times
stops_filtered = stops.copy()
stops_filtered['stop_id'] = stops_filtered['stop_id'].astype(str).apply(
    lambda x: (
        re.match(r'^((?:[^:]*:){3})', x).group(1).rstrip(':')
        if re.match(r'^((?:[^:]*:){3})', x)
        else x
    )
)
# keep only stop entry for parent station, if no parent station is given, keep a stop entry
stops_parents = stops_filtered[stops_filtered['stop_id'].str.startswith('Pat')].copy()
stops_parents['stop_id'] = stops_parents['stop_id'].apply(lambda x: x if x[0] != 'P' else x[1:])
stops_filtered = stops_filtered[stops_filtered['stop_id'].str.startswith('at') | stops_filtered['stop_id'].str.startswith('obb')]
stops_filtered = stops_filtered[~stops_filtered['stop_id'].isin(stops_parents['stop_id'])].drop_duplicates(subset=['stop_id'], keep='first')
# TODO: maybe filter out special stations e.g. obb_CP_80854 Wattens Sammelpunkt Bahnhofstraße MPREIS or Pat:42:99979_HoB

stops_filtered_final = pd.concat([stops_parents, stops_filtered])
print(stops_filtered_final.shape)

stop_times_filtered = stop_times.copy()
stop_times_filtered = stop_times_filtered[stop_times_filtered['departure_time'].between('06:00:00', '20:00:00')]
stop_times_filtered = stop_times_filtered[stop_times_filtered['stop_id'].str.startswith('at')]
# TODO: check if Parent station is in stop_times and keep those
stop_times_filtered['stop_id'] = stop_times_filtered['stop_id'].astype(str).apply(
    lambda x: (
        re.match(r'^((?:[^:]*:){3})', x).group(1).rstrip(':')
        if re.match(r'^((?:[^:]*:){3})', x)
        else x
    )
)

# display(stop_times_df_obb_mod.head())
#stop_times_filtered

(381, 9)


In [21]:
stop_times_trips = pd.merge(stop_times_filtered, routes_trips, on='trip_id', how='inner')
print(stop_times_trips.shape)
stop_times_trips.head()

(9826, 21)


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,route_id,...,shape_id,trip_headsign,trip_short_name,direction_id,block_id,agency_id,route_short_name,route_long_name,route_type,rank
0,1.TA.2-101-E-c24-1.1.R,06:49:00,06:49:00,at:44:40005,1,NaN,0,0,0.00,at:esg:101:,...,2-101-E-c24-1.1.R,Linz/Donau Urnenhain Urfahr,nan,1,NaN,1,101,Linz Urnenhain Urfahr - Linz Aichinger,3,3
1,1.TA.2-101-E-c24-1.1.R,06:50:00,06:50:00,at:44:41034,2,NaN,0,0,477.42,at:esg:101:,...,2-101-E-c24-1.1.R,Linz/Donau Urnenhain Urfahr,nan,1,NaN,1,101,Linz Urnenhain Urfahr - Linz Aichinger,3,3
2,1.TA.2-101-E-c24-1.1.R,06:51:00,06:51:00,at:44:40252,3,NaN,0,0,826.57,at:esg:101:,...,2-101-E-c24-1.1.R,Linz/Donau Urnenhain Urfahr,nan,1,NaN,1,101,Linz Urnenhain Urfahr - Linz Aichinger,3,3
3,1.TA.2-101-E-c24-1.1.R,06:53:00,06:53:00,at:44:41040,4,NaN,0,0,1451.75,at:esg:101:,...,2-101-E-c24-1.1.R,Linz/Donau Urnenhain Urfahr,nan,1,NaN,1,101,Linz Urnenhain Urfahr - Linz Aichinger,3,3
4,1.TA.2-101-E-c24-1.1.R,06:53:30,06:53:30,at:44:41033,5,NaN,0,0,1691.79,at:esg:101:,...,2-101-E-c24-1.1.R,Linz/Donau Urnenhain Urfahr,nan,1,NaN,1,101,Linz Urnenhain Urfahr - Linz Aichinger,3,3


In [22]:
stop_times_grouped = stop_times_trips.groupby(['stop_id']).agg(rank=("rank", "min"), count=("rank", "count")).reset_index().copy()

# TODO: maybe after merge with obb stations
stop_times_grouped["interval"] = stop_times_grouped["count"].apply(lambda x: 840 / (x/2) if x != 0 else 1680)
stop_times_grouped["category"] = stop_times_grouped.apply(lambda x: lookup_category(x["interval"], x["rank"]), axis=1)

stop_times_grouped.sort_values(by=['count'], ascending=False, inplace=True)

stop_times_grouped

,stop_id,rank,count,interval,category
92,at:44:41164,3,287,5.853659,2
63,at:44:41065,3,179,9.385475,2
174,at:44:45485,3,171,9.824561,2
86,at:44:41146,3,160,10.500000,3
109,at:44:41227,3,159,10.566038,3
...,...,...,...,...,...
167,at:44:42586,3,7,240.000000,-1
82,at:44:41135,3,7,240.000000,-1
121,at:44:41262,3,5,336.000000,-1
175,at:44:45817,3,2,840.000000,-1


In [23]:
# TODO: change "how" to "left" to include all stops, also ones without rank and count (nan)
stops_final = pd.merge(stops_filtered_final.drop(['zone_id', 'location_type', 'level_id', 'platform_code', 'parent_station'], axis=1), stop_times_grouped, on='stop_id', how='inner')
stops_final

,stop_id,stop_name,stop_lat,stop_lon,rank,count,interval,category
0,at:44:40005,Linz/Donau Aichinger,48.341963,14.307521,3,48,35.000000,4
1,at:44:40006,Linz/Donau Aichwiesen,48.327069,14.279054,3,48,35.000000,4
2,at:44:40019,Linz/Donau Am Bachlberg,48.330742,14.272864,3,48,35.000000,4
3,at:44:40035,Leonding ASKÖ Leonding,48.273813,14.264492,3,43,39.069767,4
4,at:44:40039,Linz/Donau Bachlbergweg,48.328162,14.276278,3,48,35.000000,4
...,...,...,...,...,...,...,...,...
190,at:44:48156,Linz/Donau Grabnerstraße,48.283318,14.275847,3,52,32.307692,4
191,at:44:48157,Leonding Kollwitzstraße,48.280562,14.278362,3,53,31.698113,4
192,at:44:48158,Linz/Donau Keferfeldstraße,48.275863,14.283330,3,52,32.307692,4
193,at:44:48159,Linz/Donau Kuefsteinerstraße,48.276138,14.286528,3,52,32.307692,4
